In [1]:
%load_ext autoreload

from pathlib import Path
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output
from rich.console import Console
from rich.table import Table
from rich import box
import io
import logging

%autoreload 2
from panoseti_interface import PanosetiRun, PFFSequence
from jax_filters import KERNELS


In [2]:
# Usage
test_out_dir = Path("out")
data_dir = Path("palomar_data")
run_dir = "obs_Palomar.start_2026-01-20T02:24:17Z.runtype_obs-test.pffd"

# 1. Load the Run
run_path = data_dir / run_dir
try:
    run = PanosetiRun(run_path)
except FileNotFoundError:
    print("Run directory not found, check path.")
    exit(1)

print("Available Products:", run.list_products())
# Expected: ['dp_ph1024.bpp_2.module_253', 'dp_ph1024.bpp_2.module_252', ...]

Available Products: ['dp_ph1024.bpp_2.module_252', 'dp_ph1024.bpp_2.module_254', 'dp_ph1024.bpp_2.module_253', 'dp_ph1024.bpp_2.module_250']


In [9]:
import ipywidgets as widgets
from rich import box

class FastVisualizer:
    def __init__(self, run_object):
        self.run_object = run_object
        self.console = Console(record=True, width=60)
        
        # --- UI Components ---
        self.out_plot = widgets.Output()
        self.out_stats = widgets.Output()
        
        # Data Selection
        self.drop_stream = widgets.Dropdown(description='Stream:', layout=widgets.Layout(width='300px'))
        self.drop_algo = widgets.Dropdown(
            options=['neighbor', 'threshold'], 
            value='neighbor', 
            description='Algorithm:',
            layout=widgets.Layout(width='200px')
        )
        
        # Parameters
        self.s_frame = widgets.IntSlider(description='Frame', layout=widgets.Layout(width='500px'))
        self.s_thresh = widgets.IntSlider(value=150, min=0, max=4096, step=10, description='Threshold')
        self.s_nmin = widgets.IntSlider(value=2, min=1, max=10, description='N Min')
        
        # Toggles
        self.chk_trig = widgets.Checkbox(value=True, description='Show Triggers (>Thresh)')
        self.chk_clump = widgets.Checkbox(value=True, description='Show Supported (Neighbors)')

        # Layout
        row1 = widgets.HBox([self.drop_stream, self.drop_algo])
        row2 = widgets.HBox([self.s_thresh, self.s_nmin])
        row3 = widgets.HBox([self.chk_trig, self.chk_clump])
        
        ctrl_box = widgets.VBox([row1, self.s_frame, row2, row3])
        display(widgets.VBox([ctrl_box, self.out_plot, self.out_stats]))
        
        # Bindings
        self.drop_stream.observe(self._on_stream_change, names='value')
        for w in [self.s_frame, self.s_thresh, self.s_nmin, self.drop_algo, self.chk_trig, self.chk_clump]:
            w.observe(self._update, names='value')
            
        # Init
        self.seq = None
        self._load_streams()

    def _load_streams(self):
        prods = [p for p in self.run_object.list_products() if 'img' in p or 'ph' in p]
        self.drop_stream.options = prods
        if prods: self.drop_stream.value = prods[0]

    def _on_stream_change(self, change):
        if not change['new']: return
        self.seq = self.run_object.get_product(change['new'])
        self.s_frame.max = len(self.seq) - 1
        self.s_frame.value = 0
        self._update(None)

    def _update(self, _):
        if self.seq is None: return
        
        # 1. Fetch Data
        idx = self.s_frame.value
        raw_batch = self.seq.get_image_array(idx, 1)
        if len(raw_batch) == 0: return
        raw_img = raw_batch[0].astype(np.int16)
        
        algo_name = self.drop_algo.value
        thresh = int(self.s_thresh.value)
        n_min = int(self.s_nmin.value)
        
        # 2. Run Algorithm
        # We manually invoke logic here to get visualization masks for both algorithms
        
        # Common: Thresholding
        trigger_mask = raw_img >= thresh
        n_tot = np.sum(trigger_mask)
        
        if algo_name == 'neighbor':
            # Run Kernel to get "supported" mask
            kernel = KERNELS['neighbor']
            # Kernel returns: (trigger_mask, supported_mask, total_triggers, n_supported, keep)
            _, supported_mask, _, n_sup, keep = kernel(jnp.array(raw_img), thresh, n_min)
            
            # Convert JAX to Numpy
            supported_mask = np.array(supported_mask)
            keep = bool(keep)
            n_sup = int(n_sup)
            
        else: # Threshold Algorithm
            # Logic: Keep if total_triggers >= n_min. No concept of "supported" pixels.
            keep = n_tot >= n_min
            supported_mask = np.zeros_like(trigger_mask) # No "neighbors" to show
            n_sup = 0
            
        # 3. Plot
        self.out_plot.clear_output(wait=True)
        with self.out_plot:
            fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
            
            # Base Image
            vmin, vmax = np.min(raw_img), np.max(raw_img)
            if vmin==vmax: vmax+=1
            im = ax.imshow(raw_img, cmap='plasma', origin='upper', vmin=vmin, vmax=vmax)
            plt.colorbar(im, ax=ax, label='ADC Value')
            
            # Overlays
            n_labeled = 0
            # Layer 1: Raw Triggers (Cyan Dots)
            if self.chk_trig.value:
                ty, tx = np.where(trigger_mask)
                ax.scatter(tx, ty, c='cyan', s=10, marker='.', alpha=0.6, label='Trigger (>Thresh)')
                n_labeled += len(ty)
            
            # Layer 2: Clumps (White Xs) - Only valid for Neighbor algo
            if self.chk_clump.value and algo_name == 'neighbor':
                sy, sx = np.where(supported_mask)
                ax.scatter(sx, sy, c='white', s=40, marker='x', linewidth=1.2, label='Neighbor Support')
                n_labeled += len(sy)

            # Metadata
            ax.set_title(f"Frame {idx} | {algo_name.upper()} | Max: {vmax}")
            if n_labeled > 0:
                ax.legend(loc='upper right', fontsize='small', framealpha=0.8)
            plt.show()

        # 4. Stats Table
        self._render_stats(algo_name, n_tot, n_sup, keep)

    def _render_stats(self, algo, n_tot, n_sup, keep):
        self.out_stats.clear_output(wait=True)
        with self.out_stats:
            color = "green" if keep else "red"
            status = "KEEP" if keep else "REJECT"
            
            table = Table(title=f"Decision: [{color}]{status}[/]", box=box.SIMPLE)
            table.add_column("Metric", style="cyan")
            table.add_column("Value", justify="right")
            table.add_column("Condition", justify="right", style="dim")
            
            table.add_row("Total Triggers", str(n_tot), f"-")
            
            if algo == 'neighbor':
                pass_c = "[green]✓[/]" if n_sup >= self.s_nmin.value else "[red]✗[/]"
                table.add_row("Supported Pixels", str(n_sup), f">= {self.s_nmin.value} {pass_c}")
            else:
                pass_c = "[green]✓[/]" if n_tot >= self.s_nmin.value else "[red]✗[/]"
                table.add_row("Threshold Check", str(n_tot), f">= {self.s_nmin.value} {pass_c}")
            
            self.console.print(table)

In [10]:
vis = FastVisualizer(run)